In [1]:
import json

import pandas as pd
import ast
import os
from tqdm import tqdm

from typing import Union, List

In [2]:
tqdm.pandas()

# Load Impact DF

In [3]:
def try_literal_eval(x):
    if isinstance(x, str):
        x = x.strip()
        if (x.startswith("[") and x.endswith("]")) or \
           (x.startswith("{") and x.endswith("}")) or \
           (x.startswith("(") and x.endswith(")")):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return x
    return x

In [4]:
dataset_dir = "/home/yishin/keith/patent_research/model_io"

In [9]:
Impact_df = pd.read_csv(os.path.join(dataset_dir, "4_Impact_Sub_GDino.csv"), encoding="utf-8")
Impact_df = Impact_df.map(try_literal_eval)

Impact_df.head()

,title,caption,image_paths,best_fig_desc,Loc_class,main_class,sub_class,first_class,crop_paths,keywords,detection_graph_json,graph_status
0,Snack food product,"The image is a rectangular shape, and it is a ...",[impact_dataset/2017/USD0787150-20170523/USD07...,{'FIG. 1': 'FIG. 1 is a perspective view of a ...,"{01-01, 11-01}",1,1,01-01,"[figure_crops/0_Snack food product/FIG_4.jpg, ...","[snack, food, candy, bar]","{""directed"": true, ""multigraph"": false, ""graph...",done
1,Ring-like input device,The image is a square-shaped drawing of a ring...,[impact_dataset/2013/USD0673953-20130108/USD06...,{'FIG. 1': 'FIG. 1 is a perspective view of a ...,"{14-02, 01-01, 11-01}",1,1,01-01,[figure_crops/1_Ring-like input device/FIG_2.j...,[ring],"{""directed"": true, ""multigraph"": false, ""graph...",done
2,Snack food,The image is a black and white drawing of a sn...,[impact_dataset/2016/USD0752316-20160329/USD07...,"{'FIG. 2': 'FIG. 2 is a perspective view.', 'F...","{01-01, 11-01}",1,1,01-01,[],[snack],NaN,skipped_no_data
3,Food bar,"The image is square-shaped, and it is a food b...",[impact_dataset/2019/USD0862830-20191015/USD08...,{'FIG. 5': 'FIG. 5 is a front elevation view o...,{01-01},1,1,01-01,"[figure_crops/3_Food bar/FIG_4.jpg, figure_cro...","[food, bar]","{""directed"": true, ""multigraph"": false, ""graph...",done
4,Bracelet,"The image is a white outline of a bracelet, wh...",[impact_dataset/2018/USD0830218-20181009/USD08...,{'FIG. 1': 'FIG. 1 is a front perspective view...,"{01-01, 11-01}",1,1,01-01,"[figure_crops/4_Bracelet/FIG_2.jpg, figure_cro...",[],NaN,skipped_no_data


# Tidying Up

In [10]:
Impact_df = Impact_df.drop(columns=["graph_status"])
Impact_df = Impact_df.drop(columns=["Loc_class"])
Impact_df = Impact_df.drop(columns=["sub_class"])
Impact_df = Impact_df.rename(columns={"first_class": "full_class"})

Impact_df.head()

,title,caption,image_paths,best_fig_desc,main_class,full_class,crop_paths,keywords,detection_graph_json
0,Snack food product,"The image is a rectangular shape, and it is a ...",[impact_dataset/2017/USD0787150-20170523/USD07...,{'FIG. 1': 'FIG. 1 is a perspective view of a ...,1,01-01,"[figure_crops/0_Snack food product/FIG_4.jpg, ...","[snack, food, candy, bar]","{""directed"": true, ""multigraph"": false, ""graph..."
1,Ring-like input device,The image is a square-shaped drawing of a ring...,[impact_dataset/2013/USD0673953-20130108/USD06...,{'FIG. 1': 'FIG. 1 is a perspective view of a ...,1,01-01,[figure_crops/1_Ring-like input device/FIG_2.j...,[ring],"{""directed"": true, ""multigraph"": false, ""graph..."
2,Snack food,The image is a black and white drawing of a sn...,[impact_dataset/2016/USD0752316-20160329/USD07...,"{'FIG. 2': 'FIG. 2 is a perspective view.', 'F...",1,01-01,[],[snack],NaN
3,Food bar,"The image is square-shaped, and it is a food b...",[impact_dataset/2019/USD0862830-20191015/USD08...,{'FIG. 5': 'FIG. 5 is a front elevation view o...,1,01-01,"[figure_crops/3_Food bar/FIG_4.jpg, figure_cro...","[food, bar]","{""directed"": true, ""multigraph"": false, ""graph..."
4,Bracelet,"The image is a white outline of a bracelet, wh...",[impact_dataset/2018/USD0830218-20181009/USD08...,{'FIG. 1': 'FIG. 1 is a front perspective view...,1,01-01,"[figure_crops/4_Bracelet/FIG_2.jpg, figure_cro...",[],NaN


# Main Class and Full Class Clarification

In [ ]:
import json

In [30]:
full_class_path = "/home/yishin/keith/patent_research/class_folder/full_locarno_classes.json"

with open(full_class_path, "r", encoding="utf-8") as file:
    full_class_dict= json.load(file)

In [31]:
main_class_path = "/home/yishin/keith/patent_research/class_folder/main_locarno_classes.json"

with open(main_class_path, "r", encoding="utf-8") as file:
    main_class_dict = json.load(file)

In [38]:
Impact_df["main_class_desc"] = (
    Impact_df["main_class"]
    .apply(
        lambda x: main_class_dict.get(str(x).strip().zfill(2))
        if pd.notna(x)
        else None
    )
)

Impact_df["full_class_desc"] = (
    Impact_df["full_class"]
    .apply(
        lambda x: full_class_dict.get(str(x).strip())
        if pd.notna(x)
        else None
    )
)

In [39]:
Impact_df.head()

,title,caption,image_paths,best_fig_desc,main_class,full_class,crop_paths,keywords,detection_graph_json,main_class_desc,full_class_desc
0,Snack food product,"The image is a rectangular shape, and it is a ...",[impact_dataset/2017/USD0787150-20170523/USD07...,{'FIG. 1': 'FIG. 1 is a perspective view of a ...,1,01-01,"[figure_crops/0_Snack food product/FIG_4.jpg, ...","[snack, food, candy, bar]","{""directed"": true, ""multigraph"": false, ""graph...",Foodstuffs,"BAKERS' PRODUCTS, BISCUITS, PASTRY, PASTA AND ..."
1,Ring-like input device,The image is a square-shaped drawing of a ring...,[impact_dataset/2013/USD0673953-20130108/USD06...,{'FIG. 1': 'FIG. 1 is a perspective view of a ...,1,01-01,[figure_crops/1_Ring-like input device/FIG_2.j...,[ring],"{""directed"": true, ""multigraph"": false, ""graph...",Foodstuffs,"BAKERS' PRODUCTS, BISCUITS, PASTRY, PASTA AND ..."
2,Snack food,The image is a black and white drawing of a sn...,[impact_dataset/2016/USD0752316-20160329/USD07...,"{'FIG. 2': 'FIG. 2 is a perspective view.', 'F...",1,01-01,[],[snack],NaN,Foodstuffs,"BAKERS' PRODUCTS, BISCUITS, PASTRY, PASTA AND ..."
3,Food bar,"The image is square-shaped, and it is a food b...",[impact_dataset/2019/USD0862830-20191015/USD08...,{'FIG. 5': 'FIG. 5 is a front elevation view o...,1,01-01,"[figure_crops/3_Food bar/FIG_4.jpg, figure_cro...","[food, bar]","{""directed"": true, ""multigraph"": false, ""graph...",Foodstuffs,"BAKERS' PRODUCTS, BISCUITS, PASTRY, PASTA AND ..."
4,Bracelet,"The image is a white outline of a bracelet, wh...",[impact_dataset/2018/USD0830218-20181009/USD08...,{'FIG. 1': 'FIG. 1 is a front perspective view...,1,01-01,"[figure_crops/4_Bracelet/FIG_2.jpg, figure_cro...",[],NaN,Foodstuffs,"BAKERS' PRODUCTS, BISCUITS, PASTRY, PASTA AND ..."


In [40]:
Impact_df.to_csv("/home/yishin/keith/patent_research/model_io/5_Impact_Sub_Final.csv", index=False, encoding="utf-8")